<a href="https://colab.research.google.com/github/PranjalAsthana/InfoRetrievalMethods/blob/main/invertedindexcompression2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import re
from collections import defaultdict

# Sample document collection
documents = {
    1: "Information retrieval is the process of finding relevant information",
    2: "Search engines use information retrieval models to rank documents"
}

def tokenize(text):
    return re.findall(r"\b[a-z0-9]+\b", text.lower())

def build_inverted_index(docs):
    """Create term -> sorted document ID postings."""
    index = defaultdict(list)

    for doc_id, text in docs.items():
        for term in set(tokenize(text)):
            index[term].append(doc_id)

    return {term: sorted(postings) for term, postings in index.items()}

def gap_encode(postings):
    """Example: [2, 5, 9] becomes [2, 3, 4]."""
    gaps = []
    previous = 0

    for doc_id in postings:
        gaps.append(doc_id - previous)
        previous = doc_id

    return gaps

def gap_decode(gaps):
    postings = []
    previous = 0

    for gap in gaps:
        previous += gap
        postings.append(previous)

    return postings

def variable_byte_encode(numbers):
    """Encode positive integers into variable-length bytes."""
    encoded = []

    for number in numbers:
        if number <= 0:
            raise ValueError("Variable Byte Encoding requires positive integers.")

        byte_list = []
        while number > 0:
            byte_list.insert(0, number % 128)
            number //= 128

        byte_list[-1] += 128  # Mark the final byte
        encoded.extend(byte_list)

    return encoded

def variable_byte_decode(encoded):
    numbers = []
    number = 0

    for byte in encoded:
        if not 0 <= byte <= 255:
            raise ValueError("Encoded values must be between 0 and 255.")

        number = number * 128 + (byte % 128)

        if byte >= 128:
            numbers.append(number)
            number = 0

    if number != 0:
        raise ValueError("Incomplete Variable Byte sequence.")

    return numbers

def compress_index(index):
    """Apply Gap Encoding followed by Variable Byte Encoding."""
    compressed = {}

    for term, postings in index.items():
        gaps = gap_encode(postings)
        compressed[term] = variable_byte_encode(gaps)

    return compressed

def decompress_postings(encoded_postings):
    gaps = variable_byte_decode(encoded_postings)
    return gap_decode(gaps)

def evaluate_and_query(query, compressed_index):
    """Evaluate an AND query using compressed postings lists."""
    terms = [
        word.lower() for word in re.findall(r"[a-zA-Z0-9]+", query)
        if word.upper() != "AND"
    ]

    if not terms:
        return []

    postings_lists = []
    for term in terms:
        if term not in compressed_index:
            return []
        postings_lists.append(
            decompress_postings(compressed_index[term])
        )

    result = set(postings_lists[0])
    for postings in postings_lists[1:]:
        result.intersection_update(postings)

    return sorted(result)

if __name__ == "__main__":
    index = build_inverted_index(documents)

    print("ORIGINAL INVERTED INDEX")
    for term in sorted(index):
        print(f"{term}: {index[term]}")

    term = "information"
    original = index[term]
    gaps = gap_encode(original)
    encoded = variable_byte_encode(gaps)
    decoded_gaps = variable_byte_decode(encoded)
    reconstructed = gap_decode(decoded_gaps)

    print("\nCOMPRESSION EXAMPLE")
    print("Term:", term)
    print("Original postings:", original)
    print("Gap encoded:", gaps)
    print("Variable Byte encoded:", encoded)
    print("Decoded gaps:", decoded_gaps)
    print("Reconstructed postings:", reconstructed)

    compressed_index = compress_index(index)

    print("\nCOMPRESSED INVERTED INDEX")
    for word in sorted(compressed_index):
        print(f"{word}: {compressed_index[word]}")

    valid = all(
        decompress_postings(compressed_index[word]) == index[word]
        for word in index
    )
    print("\nCompression/decompression verification:", valid)

    query = "information AND retrieval"
    results = evaluate_and_query(query, compressed_index)

    print("\nQUERY EVALUATION")
    print("Query:", query)
    print("Matching document IDs:", results)
    for doc_id in results:
        print(f"Document {doc_id}: {documents[doc_id]}")


ORIGINAL INVERTED INDEX
documents: [2]
engines: [2]
finding: [1]
information: [1, 2]
is: [1]
models: [2]
of: [1]
process: [1]
rank: [2]
relevant: [1]
retrieval: [1, 2]
search: [2]
the: [1]
to: [2]
use: [2]

COMPRESSION EXAMPLE
Term: information
Original postings: [1, 2]
Gap encoded: [1, 1]
Variable Byte encoded: [129, 129]
Decoded gaps: [1, 1]
Reconstructed postings: [1, 2]

COMPRESSED INVERTED INDEX
documents: [130]
engines: [130]
finding: [129]
information: [129, 129]
is: [129]
models: [130]
of: [129]
process: [129]
rank: [130]
relevant: [129]
retrieval: [129, 129]
search: [130]
the: [129]
to: [130]
use: [130]

Compression/decompression verification: True

QUERY EVALUATION
Query: information AND retrieval
Matching document IDs: [1, 2]
Document 1: Information retrieval is the process of finding relevant information
Document 2: Search engines use information retrieval models to rank documents
